<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/d3-cifar-competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 4 · Notebook 6 — The CIFAR-10 competition

Your team's job: the highest **test accuracy** on CIFAR-10, with a network you
designed. The leaderboard opens now and stays open. Day 5's
podium uses the standings at **09:00**.

**Rules**

1. Your own architecture, built from layers. **No pretrained weights** (no
   `weights=` or `pretrained=` anything). Day 5's lesson is why.
2. The saved model must be **under 40 MB**.
3. Train only on `train_set`. Tune on the **validation** set (the prelims). Run the final test cell
   (the O-level) only when you are ready to post a score, and post the line
   it prints with a screenshot. The line counts how many times you ran it.
4. Change one thing at a time and write down what it did in the log at the
   bottom. "What did you try and what happened?" is the question we ask the
   top three.
5. Your team's **best** posted score counts. Teams on a GPU train on three
   times as many photos, so CPU and GPU entries are ranked on separate lines
   of the board.

In [ ]:
import os
import hashlib
import pathlib
import shutil
import urllib.request
import time
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import torch
from torch import nn
import torchvision
from torchvision import transforms


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


def fetch_from_mirror(url, dest, sha256):
    """Download url to dest and check its sha256. Returns True if the file is good."""
    # The original data servers get slow when a whole class downloads at once,
    # so we keep a copy of each file on our own fast server.
    dest = pathlib.Path(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    def good(path):
        with open(path, "rb") as f:
            return hashlib.file_digest(f, "sha256").hexdigest() == sha256

    if dest.exists() and good(dest):
        return True
    # Download to a .part file first, so a stopped download never looks finished.
    part = dest.with_name(dest.name + ".part")
    try:
        # Our server turns away requests that look like they come from a robot,
        # so we give ours a name.
        request = urllib.request.Request(url, headers={"User-Agent": "tk-accelerator-notebooks"})
        with urllib.request.urlopen(request, timeout=30) as reply, open(part, "wb") as out:
            shutil.copyfileobj(reply, out)
        if not good(part):
            raise ValueError("the checksum does not match")
        part.replace(dest)
        return True
    except Exception:
        part.unlink(missing_ok=True)
        print("Our copy of the data did not download, so we are using the original source.")
        return False


class DataMissing(Exception):
    """Stops a cell with one plain message instead of a long error."""

    def __init__(self, message):
        super().__init__(message)
        self.message = message

    def _render_traceback_(self):
        return [self.message]


def require_data(name, *paths):
    """Stop with a plain message unless the downloaded files are on disk."""
    if not all(pathlib.Path(p).exists() for p in paths):
        raise DataMissing(f"The {name} data isn't downloaded yet. Run the ⬇️ Download cell above first.")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# The instructors' checker sets NB_CHECK to run this notebook quickly; ignore it.
QUICK = os.environ.get("NB_CHECK") == "1"
torch.manual_seed(0)
print("Running on", device, "— for a GPU: Runtime → Change runtime type → T4 GPU")

## 1. Data

Same data and split as this morning. The **test** set is the official 10,000
CIFAR-10 test photos.

🔮 **Decide as a team.** Which augmentations will you try? Add them to
`train_tf` (ideas: `transforms.RandomCrop(32, padding=4)`,
`transforms.ColorJitter(0.2, 0.2, 0.2)`). Remember: never augment the
validation or test photos.

In [ ]:
augmentations_we_will_try = ""  # @param {type:"string"}

⬇️ **Download the data (170 MB).** Run the next cell once, before the cell that loads the data. It downloads 170 MB, so skip it if you are only reading. It is safe to run twice. "Run all" runs it too.

In [ ]:
# ⬇️ Download cell: CIFAR-10 (170 MB)
import tarfile

if pathlib.Path("data/cifar-10-batches-py").exists():
    print("CIFAR-10 is already downloaded. Nothing to do.")
else:
    if fetch_from_mirror("https://data.tk.sg/cifar-10/cifar-10-python.tar.gz", "data/cifar-10-python.tar.gz",
                         "6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce"):
        # Unpack into a spare folder first, so a stopped unpack never looks finished.
        shutil.rmtree("data/cifar-unpacking", ignore_errors=True)
        with tarfile.open("data/cifar-10-python.tar.gz") as tar:
            tar.extractall("data/cifar-unpacking", filter="data")
        pathlib.Path("data/cifar-unpacking/cifar-10-batches-py").rename("data/cifar-10-batches-py")
        shutil.rmtree("data/cifar-unpacking")
    else:
        torchvision.datasets.CIFAR10("data", download=True)
    print("Done: CIFAR-10 downloaded (170 MB).")

In [ ]:
normalise = transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    # ↑ Add augmentations above this line: they work on the photo, before it
    #   becomes a tensor. Leave the next two lines last.
    transforms.ToTensor(),
    normalise,
])
plain_tf = transforms.Compose([transforms.ToTensor(), normalise])

require_data("CIFAR-10", "data/cifar-10-batches-py")
full_train = torchvision.datasets.CIFAR10("data", train=True, download=False, transform=train_tf)
full_val = torchvision.datasets.CIFAR10("data", train=True, download=False, transform=plain_tf)
test_set = torchvision.datasets.CIFAR10("data", train=False, download=False, transform=plain_tf)

order = torch.randperm(50000, generator=torch.Generator().manual_seed(0))
n_train = 1000 if QUICK else (45000 if device.type == "cuda" else 15000)
train_set = torch.utils.data.Subset(full_train, order[:n_train].tolist())
val_set = torch.utils.data.Subset(full_val, order[45000:45000 + (500 if QUICK else 5000)].tolist())
if QUICK:
    test_set = torch.utils.data.Subset(test_set, list(range(500)))

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_set, batch_size=512)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=512)
print(f"train {len(train_set)}, validation {len(val_set)}, test {len(test_set)}")

## 2. Your model

This starting model is this morning's CNN. It gets you on the board; it will
not win. Ideas, one at a time: more filters, a third convolution block,
`nn.BatchNorm2d` after each convolution, `nn.Dropout` before the last layer.
The learning rate matters as much as the layers: in our tests the same
network got 29% with `lr = 0.05` and 57% with `lr = 0.01` after four epochs.
You can also try `torch.optim.Adam(model.parameters(), lr=1e-3)`.

✏️ **Exercise 2a.** Design `make_model()`. It must return a network that turns
a batch of `3 × 32 × 32` images into 10 scores each.

In [ ]:
def make_model():
    # This morning's CNN. Improve it one change at a time.
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32 -> 16
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 16 -> 8
        nn.Flatten(),
        nn.Linear(32 * 8 * 8, 64), nn.ReLU(),
        nn.Linear(64, 10),
    )

✅ The rules check: right output shape, and under 40 MB when saved.

In [ ]:
m = make_model()
check(is_done(m), "make_model returns ... . Put a network back in it.")
check(tuple(m(torch.zeros(2, 3, 32, 32)).shape) == (2, 10), "Two images in should give 2 × 10 scores out.")
torch.save(m.state_dict(), "model_size_check.pt")
size_mb = os.path.getsize("model_size_check.pt") / 1e6
check(size_mb < 40, f"Your model is {size_mb:.1f} MB; the limit is 40 MB. Fewer filters or a smaller Linear layer.")
print(f"Model size {size_mb:.2f} MB, {sum(p.numel() for p in m.parameters()):,} numbers to learn.")

## 3. Train, and watch validation

In [ ]:
epochs = 3  # @param {type:"integer"}
lr = 0.01  # @param {type:"number"}

torch.manual_seed(0)
model = make_model().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
loss_fn = nn.CrossEntropyLoss()


def accuracy_on(loader):
    model.eval()
    right = total = 0
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images.to(device)).argmax(dim=1).cpu()
            right += (preds == labels).sum().item()
            total += len(labels)
    return right / total


assert train_loader.dataset is train_set, "Train only on train_set: train_loader must be built from it."
for epoch in range(1 if QUICK else epochs):
    started = time.time()
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(images), labels)
        loss.backward()
        optimizer.step()
    print(f"epoch {epoch + 1}: validation {accuracy_on(val_loader):.1%}  ({time.time() - started:.0f} s)")

🔮 **Predict before each run.** Write down what you changed and what you expect
validation accuracy to do. Then run, and fill in the log in section 5.

In [ ]:
my_prediction = ""  # @param {type:"string"}

## 4. The O-level: test accuracy

Run this only when you are ready to post a score. Every time you run it and
then change your model because of the result, the test set becomes a little
more like a prelim.

🔮 **Predict.** Will your test accuracy be higher or lower than your last
validation accuracy, and by how much?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
assert not set(train_set.indices) & set(order[45000:].tolist()), "Training data overlaps validation: not a valid entry."
assert len(train_set) <= 45000, "Training data must come from the first 45,000 shuffled photos."
test_runs = globals().get("test_runs", 0) + 1
test_acc = accuracy_on(test_loader)
torch.save(model.state_dict(), "entry.pt")
entry_mb = os.path.getsize("entry.pt") / 1e6
print(f"TEST ACCURACY {test_acc:.2%} | size {entry_mb:.1f} MB | device {device.type} | "
      f"trained on {len(train_set)} | epochs {epochs} | test run {test_runs} | "
      f"{datetime.now(ZoneInfo('Asia/Singapore')):%a %H:%M} SGT")
print("Post this line, a screenshot, and File → Download → .ipynb to the instructors.")

## 5. Team log

Fill this in as you go. The top three teams present it on the morning of Day 5.

| Run | What we changed | Validation before | Validation after | Keep? |
|---|---|---|---|---|
| 1 | Starting model | | | |
| 2 | | | | |
| 3 | | | | |